# Construisons un module de géométrie (1/2)

Les deux prochains TPs seront consacrés à la création d'un module pour gérer la géométrie. L'objectif est d'apprendre à fabriquer un module avec des objets (création de classes) qui vont dépendre les uns des autres.

Pour commencer, nous allons travailler dans un notebook et créer deux objets géométriques : le disque et le polygone. Lors du prochain TP, nous créerons réellement le module.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
mpl.rcParams['savefig.dpi'] = 80

## La classe `disque`

Commençons par créer une classe `disque` qui va nous permettre de manipuler des disques en connaissant leur centre et leur rayon. L'objectif est ici simplement de manipuler une nouvelle classe.

**Question 1**

> 1. Créez une classe `disque` dont on pourra initialiser une nouvelle instance en donnant en argument de la fonction `__init__` le centre `c`, le rayon `r` et son nom `nom`. Vous veillerez à ce que
>   - `c` puisse être une `list` ou un `ndarray` contenant des `float`;
>   - `c` soit de taille 2 ;
>   - `r` soit un réel (ou un entier) strictement positif ;
>   - `nom` puisse ne pas être renseigné et vaudra par défaut `"D"`.
> 2. Un objet de la classe `disque` aura alors 3 attributs :
>   - `_c` un `ndarray` de taille 2 contenant le centre du disque ;
>   - `_r` un `float` contenant le rayon du disque ;
>   - `_name` une chaîne de caractères contenant le nom du disque.
> 3. Vérifiez que votre classe fonctionne pour le moment...

**Question 2**

> 1. Ajoutez une fonction membre `centre` qui retourne le `ndarray` `_c`. Décorez cette fonction avec le décorateur `@property` (attention, la classe `disque` doit pour cela hériter de la classe `object`) afin que la fonction devienne un attribut accessible seulement en lecture.
> 2. Ajoutez de la même façon un attribut en lecture seule `rayon` qui renvoie le rayon du disque.

**Question 3**

> Ajoutez trois attributs en lecture seule (fonction décorée avec `@property`) qui retourne le périmètre, l'aire et le centre de gravité du disque.

*Indication : la valeur $\pi$ pourrait être un nombre partagé par toutes les instances de la classe `disque`*

**Question 4**

> Ajoutez un attribut en lecture seule (une fonction décorée avec `@property`) qui est un dictionnaire de la forme
> ```python
> {
>   'x': [xmin, xmax],
>   'y': [ymin, ymax],
> }
> ```
> où `xmin` et `xmax` sont une borne inférieure et une borne supérieure des abscisses des points du disque et `ymin` et `ymax` des ordonnées.

**Question 5**

> Ajoutez une fonction membre `info` qui retourne une chaine de caractères contenant les informations du disque en respectant ce format
> ```
> ---------------------------------------------------------------
> Disque D de centre [0. 0.] et de rayon 1
>     Le périmètre de D vaut        6.283185307179586
>     L'aire de D vaut              3.141592653589793
>     Le centre de gravité de D est [0. 0.]
> ---------------------------------------------------------------
> ```


**Question 6**

> 1. Ajoutez une fonction membre `_is_in` qui prend en arguments deux `ndarray` (notés `x` et `y`) de même forme et qui retourne un `ndarray` de booléens de la même forme dont les valeurs sont `True` ou `False` selon que le point appartient ou non au disque.
> 2. Ajoutez une fonction membre `visualize` qui prend en arguments un axe `ax` de la classe `matplotlib.axes._subplots.AxesSubplot` et qui affiche un nuage de points équi-répartis à l'intérieur du disque. Vous pourrez essayer de reproduire cette figure obtenue pour le cercle de centre $(0, 0)$ et de rayon 1 :
![cercle](cercle.png)

## Le module `geometrie`

**Question 7**

Nous allons commencer à construire le module `geometrie` avec la classe `disque` précédemment construite.

> 1. Créez un dossier `TP07_geometrie` dans le répertoire à côté du notebook.
> 2. Créez un fichier `__init__.py` dans ce dossier (ce sera le fichier d'initialisation du module).
> 3. Créez un fichier `disque.py` dans ce dossier et copier le code de la classe `disque` à l'intérieur.
> 4. Ajoutez dans le fichier `disque.py` les entêtes nécessaires, par exemple `import numpy as np`.
> 5. Ajoutez dans le fichier `__init__.py` la ligne
> ```python
> from .disque import disque
> ```

A présent, vous devez pour accéder à votre module nouvellement créé. Testez pour cela les lignes de code des cellules suivantes.

In [ ]:
import TP07_geometrie as geom

In [ ]:
D = geom.disque([0, 0], 1, "D")
print(D.info())

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(1, 1, 1)
ax.grid(False)
D.visualize(ax)

## Une deuxième classe : la classe `polygone`

Nous allons à présent ajouter un deuxième objet géométrie de type polygone. Un polygone sera défini par la liste de ses sommets (nous utiliserons un `ndarray` de forme $(p,2)$ où $p$ est le nombre de sommets pour les stocker).

*NB : pour simplifier les choses, nous supposerons dans la suite que tous les polygones que l'on construira seront convexes.*

Quelques rappels de géométrie sur les polygones convexes. Nous notons $s_0, \ldots, s_{p-1}$, les $p$ sommets de ce polygone. L'aire d'un tel polygone vaut
$$
A = \frac{1}{2} \sum_{k=1}^{p-2} (s_{k,x}-s_{0,x})(s_{k+1,y}-s_{0,y}) - (s_{k,y}-s_{0,y})(s_{k+1,x}-s_{0,x})
$$
Par ailleurs, pour déterminer si un point $P$ de coordonnées $(x, y)$ est dans un tel polynômes, il faut calculer $\theta$ l'angle défini par
$$
\theta = \sum_{k=0}^{p-2} \widehat{s_k P s_{k+1}},
\quad\text{avec}\quad
\widehat{s_k P s_{k+1}} = \operatorname{arctan2} (\operatorname{det}, \operatorname{pscal})
$$
et
$$
\begin{aligned}
\operatorname{det} &= u_x v_y - u_y v_x,&
\operatorname{pscal} &= u_x v_x + u_y v_y,\\
u_x &= s_{k,x} - x,&
u_y &= s_{k,y} - y,\\
v_x &= s_{k+1,x} - x,&
v_y &= s_{k+1,y} - y.
\end{aligned}
$$

**Question 8**

> Créez la classe `polygone` que vous mettrez dans un fichier `polygone.py` dans le même répertoire que `disque.py` en adaptant la classe `disque`. Pensez à ajouter la ligne 
> ```python
> from .polygone import polygone
> ```
> dans le fichier `__init__.py`.
>
> Testez ensuite cette nouvelle classe en créant et affichantun triangle et un carré de votre choix (vous pourrez aussi faire une figure avec plusieurs formes géométriques.
>
> *(Question bonus)* en utilisant `ax.get_xlim` et `ax.get_ylim`, récupérez et modifiez les bornes du graphique afin que les figures affichent bien tous les objets géométriques que vous voulez.